# 🏆 HAIDER Benchmark v7 — GRAND MASTER EDITION

## ROOT CAUSE FIX (Why v6 failed)
```
v6 problem:  context_size=2048, max_tokens=1024
             Reasoning model needs 3000-5000 tokens to THINK.
             Model got cut off mid-reasoning → wrong answers.
             104s timeouts everywhere = model thinking but truncated.

v7 fix:      context_size=8192, max_tokens=4096 (no cutoff)
             No per-question timeout (reasoning models need time)
             Smarter question selection (quality over quantity)
             Global 11h 20min = 40,800s hard stop
```

## Design
| Dataset | N | Priority | Purpose |
|---------|---|----------|---------|
| MATH (Hendrycks) | 80 | HIGH | Competition math — prestige benchmark |
| GSM8K (hard only) | 40 | MEDIUM | Arithmetic reasoning |
| MMLU-Math | 40 | MEDIUM | MCQ coverage |
| **Total** | **160** | | **~8-10h at 11.5 t/s** |

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: BOOTSTRAP (same as v5/v6, unchanged)
# ═══════════════════════════════════════════════════════════
import os, sys, subprocess, importlib, site

os.environ['GGML_CUDA_NO_GRAPHS']    = '1'
os.environ['LLAMA_CPP_LOG_LEVEL']    = '2'
os.environ['CUDA_VISIBLE_DEVICES']   = '0,1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def check_cuda_support():
    try:
        from llama_cpp import llama_supports_gpu_offload
        return llama_supports_gpu_offload()
    except Exception:
        return False

print('📦 Checking llama_cpp CUDA build...', end=' ')
needs_install = False
try:
    from llama_cpp import Llama
    cuda_ok = check_cuda_support()
    if cuda_ok:
        print('✅ already installed with CUDA')
    else:
        print('⚠️  no CUDA — reinstalling...')
        needs_install = True
except ImportError:
    print('not found — installing...')
    needs_install = True

if needs_install:
    print('   Trying pre-built CUDA 12.1 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', 'llama-cpp-python',
        '--upgrade', '--quiet', '--user', '--force-reinstall',
        '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu121'
    ], capture_output=True, text=True)
    importlib.invalidate_caches()
    for d in site.getsitepackages() + [site.getusersitepackages()]:
        if d not in sys.path: sys.path.insert(0, d)
    if not check_cuda_support():
        print('   Building from source with CUDA sm_75 (T4)...')
        os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75'
        os.environ['FORCE_CMAKE'] = '1'
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        'llama-cpp-python', '--upgrade', '--user', '--force-reinstall'])
        importlib.invalidate_caches()
    from llama_cpp import Llama
    print('   ✅ llama_cpp installed')

for pkg, imp in [('datasets','datasets'),('tqdm','tqdm'),('pandas','pandas'),('scipy','scipy')]:
    try: __import__(imp); print(f'   ✅ {pkg}')
    except ImportError:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet','--user'],capture_output=True)
        importlib.invalidate_caches()
        try: __import__(imp); print(f'   ✅ {pkg}')
        except: print(f'   ❌ {pkg}')

gpu_info = subprocess.run(
    ['nvidia-smi','--query-gpu=index,name,memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True)
if gpu_info.returncode == 0:
    print('\n🖥️  GPUs:')
    for line in gpu_info.stdout.strip().split('\n'): print(f'   {line.strip()}')

print('\n✅ Bootstrap complete')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: CONFIGURATION — GRAND MASTER SETTINGS
# ═══════════════════════════════════════════════════════════
import os, pathlib
from huggingface_hub import hf_hub_download

# ── Model ────────────────────────────────────────────────
HF_REPO_ID  = 'ABDUL-HASEEB-TANOLI/HAIDER-Math-32B-v1-GGUF'
HF_FILENAME = 'HAIDER-Math-32B-v1-Q4_K_M.gguf'
HF_TOKEN    = None

print('⬇️ Downloading model...')
MODEL_PATH = hf_hub_download(
    repo_id=HF_REPO_ID, filename=HF_FILENAME,
    token=HF_TOKEN, cache_dir='/kaggle/working/hf_cache')
print(f'✅ Model ready: {pathlib.Path(MODEL_PATH).stat().st_size/1e9:.1f} GB')

# ── GPU Config ───────────────────────────────────────────
GPU_SPLIT       = [0.5, 0.5]
N_GPU_LAYERS    = -1

# ╔══════════════════════════════════════════════════╗
# ║  THE FIX: context=8192, not 2048                ║
# ║  Reasoning model needs space to THINK           ║
# ╚══════════════════════════════════════════════════╝
CONTEXT_SIZE   = 8192   # was 2048 — THIS WAS THE KILLER
BATCH_SIZE     = 512    # reduced (larger ctx needs more VRAM)
N_THREADS      = 1
N_THREADS_BATCH= 8

# ── Inference ────────────────────────────────────────────
TEMPERATURE    = 0.0    # greedy — reproducible
TOP_P          = 1.0
REPEAT_PENALTY = 1.05
STOP_SEQUENCES = ['<|end|>', '<|eot_id|>', '<|im_end|>', '</s>']

# ── Max tokens: REASONING MODEL NEEDS SPACE ────────────
# v6 had 1024 for GSM8K — model cut off thinking at 500 tokens
# v7: 4096 tokens — model can fully complete chain-of-thought
MAX_TOKENS_BY_TASK = {
    'math':   4096,   # competition math: needs long reasoning
    'gsm8k':  3072,   # arithmetic: still needs full thinking
    'mmlu':    512,   # MCQ: short answer (A/B/C/D + brief reason)
    'default': 2048,
}

# ── Question counts (quality over quantity) ──────────────
N_MATH      = 80    # HIGH PRIORITY — prestige benchmark
N_GSM8K     = 40    # select harder questions only
N_MMLU      = 10    # per subject × 4 = 40 total
RAND_SEED   = 42

# ── Global session timer ──────────────────────────────────
# Kaggle: 11h 50min actual. Auto-stop at 11h 20min = 40,800s
SESSION_LIMIT_SEC = 11 * 3600 + 20 * 60   # 40,800 seconds

# ── Paths ────────────────────────────────────────────────
os.makedirs('/kaggle/working', exist_ok=True)
CHECKPOINT_FILE = '/kaggle/working/gm_checkpoint.json'
RESULTS_FILE    = '/kaggle/working/gm_results.json'

PAPER_META = {
    'model_id'     : HF_REPO_ID,
    'model_file'   : HF_FILENAME,
    'quantization' : 'Q4_K_M (4-bit, GGUF)',
    'temperature'  : TEMPERATURE,
    'context_size' : CONTEXT_SIZE,
    'n_math'       : N_MATH,
    'n_gsm8k'      : N_GSM8K,
    'n_mmlu'       : N_MMLU * 4,
    'rand_seed'    : RAND_SEED,
    'hardware'     : '2x NVIDIA Tesla T4 (15GB each)',
    'version'      : 'Grand Master v7',
}

print('\n✅ Configuration ready (GRAND MASTER MODE)')
print(f'   Context size   : {CONTEXT_SIZE} tokens (was 2048 — now FIXED)')
print(f'   Max tokens MATH: {MAX_TOKENS_BY_TASK["math"]} (was 2048)')
print(f'   Max tokens GSM8K:{MAX_TOKENS_BY_TASK["gsm8k"]} (was 1024)')
print(f'   Temperature    : {TEMPERATURE} (greedy)')
print(f'   Questions      : {N_MATH} MATH + {N_GSM8K} GSM8K + {N_MMLU*4} MMLU = {N_MATH+N_GSM8K+N_MMLU*4}')
print(f'   Session limit  : {SESSION_LIMIT_SEC//3600}h {(SESSION_LIMIT_SEC%3600)//60}min')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: LOAD MODEL + GPU VERIFY + SPEED TEST
# ═══════════════════════════════════════════════════════════
import time, subprocess
from llama_cpp import Llama

print(f'⏳ Loading model with context={CONTEXT_SIZE}...')
print('   (Larger context = more VRAM used — expected)')
t0 = time.time()

llm = Llama(
    model_path      = MODEL_PATH,
    n_gpu_layers    = N_GPU_LAYERS,
    n_ctx           = CONTEXT_SIZE,
    n_batch         = BATCH_SIZE,
    n_threads       = N_THREADS,
    n_threads_batch = N_THREADS_BATCH,
    tensor_split    = GPU_SPLIT,
    flash_attn      = True,
    use_mmap        = True,
    use_mlock       = False,
    verbose         = False,
)
print(f'✅ Model loaded in {time.time()-t0:.1f}s')

# VRAM check
print('\n📊 GPU VRAM after load:')
mem = subprocess.run(
    ['nvidia-smi','--query-gpu=index,memory.used,memory.free','--format=csv,noheader'],
    capture_output=True, text=True)
total_used = 0
for i, line in enumerate(mem.stdout.strip().split('\n')):
    print(f'   GPU {i}: {line.strip()}')
    try: total_used += int(line.strip().split(',')[1].replace('MiB','').strip())
    except: pass
print(f'   Total VRAM used: {total_used} MiB ({total_used/1024:.1f} GB)')
if total_used < 8000:
    print('   🚨 WARNING: Low VRAM — model on CPU! Re-run Cell 1.')
elif total_used < 18000:
    print('   ⚠️  Partial GPU (some layers on CPU)')
else:
    print('   ✅ Full GPU utilization')

# Warmup
print('\n🔥 Warmup...')
_ = llm('<|user|>2+2<|end|><|assistant|>', max_tokens=32)

# Speed test (with larger context — speed may differ slightly)
print('\n🔬 Speed test...')
tps_list = []
for p in [
    '<|user|>What is 12 + 15? Show your work.<|end|><|assistant|>',
    '<|user|>Solve: x^2 - 5x + 6 = 0. Show all steps.<|end|><|assistant|>',
    '<|user|>A store sells apples at $2 each. If you buy 7 apples, how much do you pay?<|end|><|assistant|>',
]:
    t0 = time.time()
    out = llm(p, max_tokens=200, temperature=0.0, echo=False)
    elapsed = time.time() - t0
    toks = out['usage']['completion_tokens']
    tps = toks / max(elapsed, 0.001)
    tps_list.append(tps)
    print(f'   {tps:.1f} t/s ({toks} tok in {elapsed:.1f}s)')

avg_tps = sum(tps_list) / len(tps_list)
print(f'\n📊 Avg speed: {avg_tps:.1f} t/s')

# Time estimates with new settings
# Reasoning model: MATH averages ~300 tokens/s thinking = ~300s per question
# at 11.5 t/s that's 3000 tokens = 260s per MATH question
# GSM8K: ~150s per question (2048 tokens)
est_math  = N_MATH  * 2500 / max(avg_tps, 0.5)   # avg 2500 tokens per MATH
est_gsm8k = N_GSM8K * 1500 / max(avg_tps, 0.5)   # avg 1500 tokens per GSM8K
est_mmlu  = N_MMLU*4 * 200 / max(avg_tps, 0.5)   # avg 200 tokens per MMLU
est_total = est_math + est_gsm8k + est_mmlu

print(f'\n⏰ Time estimates at {avg_tps:.1f} t/s:')
print(f'   MATH   {N_MATH}q  (avg 2500 tok): ~{est_math/3600:.1f}h')
print(f'   GSM8K  {N_GSM8K}q  (avg 1500 tok): ~{est_gsm8k/3600:.1f}h')
print(f'   MMLU   {N_MMLU*4}q  (avg 200 tok):  ~{est_mmlu/3600:.2f}h')
print(f'   TOTAL                            : ~{est_total/3600:.1f}h')
print(f'   SESSION LIMIT                    : {SESSION_LIMIT_SEC/3600:.2f}h')
if est_total > SESSION_LIMIT_SEC:
    adj = int(SESSION_LIMIT_SEC * N_MATH / est_total * 0.9)
    print(f'\n⚠️  Estimate exceeds limit. Will auto-stop at {SESSION_LIMIT_SEC/3600:.1f}h.')
    print(f'   Questions will be answered until time runs out.')
else:
    print(f'\n✅ Fits within {SESSION_LIMIT_SEC/3600:.1f}h limit.')

PAPER_META['avg_tps'] = round(avg_tps, 2)
PAPER_META['est_total_hours'] = round(est_total/3600, 2)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: SMART DATASET LOADER
# Priority: MATH > GSM8K (harder) > MMLU
# ═══════════════════════════════════════════════════════════
from datasets import load_dataset
import random
random.seed(RAND_SEED)

def safe_load(name, fn):
    try:
        items = fn()
        print(f'  ✅ {name}: {len(items)} questions')
        return items
    except Exception as e:
        print(f'  ❌ {name}: {type(e).__name__}: {str(e)[:80]}')
        return []

# ── MATH: All 7 subjects, prioritize higher difficulty ────
def load_math():
    subjects = ['algebra','counting_and_probability','geometry',
                'intermediate_algebra','number_theory','prealgebra','precalculus']
    all_items = []
    for subj in subjects:
        try:
            ds = list(load_dataset('EleutherAI/hendrycks_math', subj, split='test'))
            for r in ds:
                level = str(r.get('level', '?'))
                all_items.append({
                    'question': r['problem'],
                    'answer'  : r['solution'],
                    'source'  : 'math',
                    'level'   : level,
                    'subject' : subj,
                    # Priority score: harder levels first
                    '_priority': int(level.replace('Level ','').replace('?','3'))
                })
            print(f'     ✅ math/{subj}: {len(ds)}q')
        except Exception as e:
            print(f'     ⚠️  math/{subj}: {str(e)[:60]}')

    if not all_items: return []

    # Sort by difficulty descending (Level 5 first), then shuffle within levels
    all_items.sort(key=lambda x: -x['_priority'])

    # Select balanced across levels: prioritize L4+L5 but include all
    from collections import defaultdict
    by_level = defaultdict(list)
    for item in all_items:
        by_level[item['level']].append(item)
    for k in by_level:
        random.shuffle(by_level[k])

    selected = []
    # Level 5: 35 questions (most challenging)
    # Level 4: 25 questions
    # Level 3: 12 questions
    # Level 1+2: 8 questions (baseline)
    level_targets = {'Level 5': 35, 'Level 4': 25, 'Level 3': 12,
                     'Level 2': 5, 'Level 1': 3}
    for lvl, target in level_targets.items():
        pool = by_level.get(lvl, [])
        selected.extend(pool[:target])

    selected = selected[:N_MATH]
    random.shuffle(selected)

    print(f'     → Pool: {len(all_items)} | Selected: {len(selected)}')
    lvl_counts = defaultdict(int)
    for item in selected: lvl_counts[item['level']] += 1
    for k in sorted(lvl_counts): print(f'       {k}: {lvl_counts[k]}')
    return selected

# ── GSM8K: Select HARDER questions (longer question text = harder) ──
def load_gsm8k():
    ds = list(load_dataset('gsm8k', 'main', split='test'))
    # Harder questions tend to be longer — use length as proxy for difficulty
    ds_with_len = [(len(r['question']), r) for r in ds]
    ds_with_len.sort(key=lambda x: -x[0])  # longest first (harder)
    # Take top 60% by length, then sample randomly from that pool
    hard_pool = [r for _, r in ds_with_len[:int(len(ds)*0.6)]]
    random.shuffle(hard_pool)
    selected = hard_pool[:N_GSM8K]
    return [{
        'question': r['question'],
        'answer'  : r['answer'].split('####')[-1].strip(),
        'source'  : 'gsm8k',
    } for r in selected]

# ── MMLU: cais/mmlu (confirmed working) ──────────────────────
def load_mmlu():
    subjects = ['elementary_mathematics','high_school_mathematics',
                'college_mathematics','abstract_algebra']
    labels = ['A','B','C','D']
    items  = []
    for subj in subjects:
        try:
            ds = list(load_dataset('cais/mmlu', subj, split='test'))
            random.shuffle(ds)
            for r in ds[:N_MMLU]:
                choices = '  '.join(f'({labels[i]}) {c}' for i,c in enumerate(r['choices']))
                items.append({
                    'question': f"{r['question']}\n{choices}",
                    'answer'  : labels[r['answer']],
                    'source'  : f'mmlu_{subj}',
                })
        except Exception as e:
            print(f'     ⚠️  mmlu/{subj}: {str(e)[:60]}')
    return items

print('📚 Loading datasets (quality-first selection)...')
all_questions = []

# Load in priority order: MATH first (most important)
math_q   = safe_load('MATH (competition)', load_math)
gsm8k_q  = safe_load('GSM8K (hard subset)', load_gsm8k)
mmlu_q   = safe_load('MMLU-Math (4 subj)', load_mmlu)

# Order: MATH first, then GSM8K, then MMLU
# This ensures if session times out, we have the most important results first
all_questions.extend(math_q)
all_questions.extend(gsm8k_q)
all_questions.extend(mmlu_q)

if not all_questions:
    raise RuntimeError('❌ No datasets loaded!')

from collections import Counter
src_counts = Counter(q['source'] for q in all_questions)
print(f'\n📊 Total: {len(all_questions)} questions (MATH first — priority order)')
for src, cnt in sorted(src_counts.items()):
    print(f'   {src:<42} {cnt}')
print()
print('💡 Questions run in this order: MATH → GSM8K → MMLU')
print('   If session times out, most important results (MATH) are preserved.')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: REASONING-AWARE ENGINE
# Key: NO per-question SIGALRM timeout.
# Reasoning models NEED to finish their chain-of-thought.
# The global timer (cell 6) will stop the loop if needed.
# ═══════════════════════════════════════════════════════════
import json, re, os, time
from datetime import datetime

# ── Prompt templates ──────────────────────────────────────
def make_prompt(question: str, source: str) -> str:
    if source.startswith('mmlu'):
        sys_msg = (
            'You are an expert mathematician. '
            'Think briefly, then answer with ONLY the letter A, B, C, or D. '
            'End your response with: Answer: [letter]'
        )
    elif source == 'gsm8k':
        sys_msg = (
            'You are an expert mathematician. '
            'Solve step-by-step showing your full reasoning. '
            'End your answer with: The answer is [number].'
        )
    else:  # math (competition)
        sys_msg = (
            'You are an expert mathematician. '
            'Solve this competition math problem with complete, rigorous reasoning. '
            'Put your final answer inside \\boxed{}.'
        )
    return (
        f'<|system|>{sys_msg}<|end|>\n'
        f'<|user|>{question}<|end|>\n'
        f'<|assistant|>'
    )

# ── Answer extraction (reasoning-model aware) ─────────────
def extract_boxed(text: str):
    """Extract last \\boxed{...} handling nested braces."""
    # Match boxed with up to 2 nesting levels
    patterns = [
        r'\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}',
        r'\\boxed\{([^}]*)\}',
    ]
    for pat in patterns:
        matches = re.findall(pat, text)
        if matches:
            return matches[-1].strip()
    return None

def normalize(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r'\s+', '', s)
    s = s.replace(',','').replace('$','').replace('\\','').replace('{','').replace('}','')
    # Normalize fractions: ½ → 0.5
    try:
        # Try to parse as number for numeric comparison
        from fractions import Fraction
        return str(float(Fraction(s)))
    except:
        return s

def score_answer(predicted: str, ground_truth: str, source: str) -> bool:
    pred = predicted.strip()
    gt   = ground_truth.strip()

    if source == 'math':
        # Extract \boxed{} from both
        pred_box = extract_boxed(pred)
        gt_box   = extract_boxed(gt)
        if pred_box and gt_box:
            return normalize(pred_box) == normalize(gt_box)
        # Fallback: check if gt_box appears in prediction
        if gt_box and normalize(gt_box) in normalize(pred):
            return True
        # Last resort: direct string match
        return normalize(gt) in normalize(pred)

    elif source == 'gsm8k':
        # Look for 'The answer is X' pattern first
        m = re.search(r'[Tt]he answer is[:\s]+([\$]?-?[\d,\.]+)', pred)
        if m:
            extracted = m.group(1).replace('$','').replace(',','')
            return extracted.strip() == gt.replace(',','')
        # Fallback: last number in response
        nums = re.findall(r'-?\d[\d,]*\.?\d*', pred.replace(',',''))
        if nums:
            return nums[-1] == gt.replace(',','')
        return False

    elif source.startswith('mmlu'):
        # Look for 'Answer: X' pattern first
        m = re.search(r'[Aa]nswer:\s*([A-Da-d])', pred)
        if m:
            return m.group(1).upper() == gt.upper()
        # Fallback: first letter mentioned
        m = re.search(r'\b([A-Da-d])\b', pred[:100])
        return bool(m) and m.group(1).upper() == gt.upper()

    return gt.lower() in pred.lower()

# ── Generation (NO per-question timeout!) ────────────────
def generate(prompt: str, source: str) -> tuple:
    """Returns (text, tokens_used, elapsed_sec)"""
    task_key  = 'math' if source=='math' else ('gsm8k' if source=='gsm8k' else 'mmlu')
    max_toks  = MAX_TOKENS_BY_TASK.get(task_key, MAX_TOKENS_BY_TASK['default'])
    t0 = time.time()
    try:
        out = llm(
            prompt,
            max_tokens     = max_toks,
            temperature    = TEMPERATURE,
            top_p          = TOP_P,
            repeat_penalty = REPEAT_PENALTY,
            stop           = STOP_SEQUENCES,
            echo           = False,
        )
        text  = out['choices'][0]['text'].strip()
        toks  = out['usage']['completion_tokens']
        # Check if model hit the limit (potential truncation)
        finish_reason = out['choices'][0].get('finish_reason', 'unknown')
        return text, toks, round(time.time()-t0, 2), finish_reason
    except Exception as e:
        return f'[ERROR:{str(e)[:80]}]', 0, round(time.time()-t0, 2), 'error'

# ── Checkpoint ───────────────────────────────────────────
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            d = json.load(f)
        print(f'♻️  Resuming: {d["completed"]} questions done')
        return d['results'], d['completed']
    return [], 0

def save_checkpoint(results, completed, stopped_early=False):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({
            'results': results,
            'completed': completed,
            'stopped_early': stopped_early,
            'saved_at': datetime.now().isoformat(),
            'paper_meta': PAPER_META
        }, f, indent=2)

print('✅ Engine ready (GRAND MASTER — no truncation)')
print(f'   Max tokens: MATH={MAX_TOKENS_BY_TASK["math"]} | GSM8K={MAX_TOKENS_BY_TASK["gsm8k"]} | MMLU={MAX_TOKENS_BY_TASK["mmlu"]}')
print(f'   Per-question timeout: NONE (model finishes its full reasoning)')
print(f'   Global stop: {SESSION_LIMIT_SEC/3600:.2f}h from bench start')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: RUN BENCHMARK — GRAND MASTER
# ═══════════════════════════════════════════════════════════
import time
from tqdm.notebook import tqdm
from collections import defaultdict
from datetime import timedelta

SAVE_EVERY = 5   # save every 5 questions (smaller value = safer)

results, start_idx = load_checkpoint()
questions_to_run   = all_questions[start_idx:]

if not questions_to_run:
    print('✅ All questions already done.')
else:
    print(f'▶️  Starting from Q{start_idx} | Remaining: {len(questions_to_run)}')
    print(f'   Total: {len(all_questions)} | Session limit: {SESSION_LIMIT_SEC/3600:.2f}h')

correct_by_src = defaultdict(int)
total_by_src   = defaultdict(int)
for r in results:
    total_by_src[r['source']] += 1
    if r['correct']: correct_by_src[r['source']] += 1

elapsed_times = []
truncated_count = 0
errors = 0
bench_start = time.time()
stopped_early = False

pbar = tqdm(questions_to_run, initial=start_idx,
            total=len(all_questions), desc='🏆 GrandMaster', unit='q')

for i, item in enumerate(pbar):
    # ── Global session guard ─────────────────────────────
    session_elapsed = time.time() - bench_start
    if session_elapsed >= SESSION_LIMIT_SEC:
        print(f'\n⏰ {SESSION_LIMIT_SEC/3600:.2f}h limit reached at Q{start_idx+i}. Stopping.')
        stopped_early = True
        break

    q_idx  = start_idx + i
    prompt = make_prompt(item['question'], item['source'])

    predicted, tokens_used, elapsed, finish_reason = generate(prompt, item['source'])

    # Flag potential truncations
    was_truncated = (
        finish_reason == 'length' and
        tokens_used >= MAX_TOKENS_BY_TASK.get(
            'math' if item['source']=='math' else
            'gsm8k' if item['source']=='gsm8k' else 'mmlu', 512) - 10
    )
    if was_truncated:
        truncated_count += 1
    if predicted.startswith('[ERROR'):
        errors += 1

    correct = score_answer(predicted, item['answer'], item['source'])

    rec = {
        'idx'          : q_idx,
        'source'       : item['source'],
        'question'     : item['question'][:400],
        'ground_truth' : item['answer'][:300],
        'predicted'    : predicted[:2000],  # store more of reasoning
        'correct'      : correct,
        'time_sec'     : elapsed,
        'tokens_used'  : tokens_used,
        'finish_reason': finish_reason,
        'truncated'    : was_truncated,
    }
    if 'level'   in item: rec['level']   = item['level']
    if 'subject' in item: rec['subject'] = item['subject']
    results.append(rec)

    total_by_src[item['source']]   += 1
    if correct: correct_by_src[item['source']] += 1
    elapsed_times.append(elapsed)

    done     = len(results)
    overall  = sum(1 for r in results if r['correct']) / done * 100
    avg_t    = sum(elapsed_times[-5:]) / len(elapsed_times[-5:])
    eta      = str(timedelta(seconds=int((len(all_questions)-done)*avg_t)))
    time_left= str(timedelta(seconds=int(SESSION_LIMIT_SEC-session_elapsed)))

    pbar.set_postfix({
        'acc': f'{overall:.1f}%',
        'avg_s': f'{avg_t:.0f}s',
        'tok': f'{tokens_used}',
        'ETA': eta,
        'left': time_left,
        'trunc': truncated_count,
    })

    if (q_idx + 1) % SAVE_EVERY == 0:
        save_checkpoint(results, q_idx + 1, stopped_early)

save_checkpoint(results, start_idx + len(questions_to_run) if not stopped_early else start_idx+i, stopped_early)
with open(RESULTS_FILE, 'w') as f:
    json.dump({'results': results, 'paper_meta': PAPER_META,
               'stopped_early': stopped_early}, f, indent=2)

total_time = time.time() - bench_start
print(f'\n✅ Done in {timedelta(seconds=int(total_time))}')
print(f'   Completed    : {len(results)} questions')
print(f'   Truncations  : {truncated_count} (these answers may be incomplete)')
print(f'   Errors       : {errors}')
if stopped_early:
    print(f'⚠️  Stopped early — {len(results)}/{len(all_questions)} done')
    print(f'   MATH completed: {total_by_src.get("math",0)}/{N_MATH}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: PAPER-READY RESULTS + TRUNCATION AUDIT
# ═══════════════════════════════════════════════════════════
import pandas as pd, json, math
from scipy import stats

if 'results' not in dir() or not results:
    with open(RESULTS_FILE) as f:
        d = json.load(f)
    results = d['results']

df = pd.DataFrame(results)

def wilson_ci(correct, n, z=1.96):
    if n == 0: return 0.0, 0.0
    p = correct / n
    denom = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denom
    margin = z * math.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return round((centre-margin)*100, 1), round((centre+margin)*100, 1)

print('=' * 70)
print('  🏆  HAIDER-Math-32B-v1 — GRAND MASTER BENCHMARK RESULTS')
print('=' * 70)
print(f'  Model        : {PAPER_META["model_id"]}')
print(f'  Quantization : {PAPER_META["quantization"]}  ← disclose in paper')
print(f'  Context      : {PAPER_META["context_size"]} tokens')
print(f'  Temperature  : {PAPER_META["temperature"]} (greedy decoding)')
print(f'  Hardware     : {PAPER_META["hardware"]}')
print(f'  Rand seed    : {PAPER_META["rand_seed"]}')
print()
print(f'  Total Qs     : {len(df)}')
print(f'  Overall acc  : {df["correct"].mean()*100:.1f}%')
print(f'  Avg time/q   : {df["time_sec"].mean():.0f}s')
print(f'  Avg tokens/q : {df["tokens_used"].mean():.0f}')
if 'truncated' in df.columns:
    trunc_n = df['truncated'].sum()
    print(f'  Truncated    : {trunc_n} ({trunc_n/len(df)*100:.1f}%)')
    if trunc_n > 0:
        print(f'  ⚠️  {trunc_n} answers may be incomplete — consider these inconclusive')
print()

# Group mmlu_ subjects
df['src_group'] = df['source'].apply(
    lambda s: 'mmlu_math (all subjects)' if s.startswith('mmlu') else s)

print(f'  {"Dataset":<38} {"N":>5}  {"Acc%":>6}  {"95% CI":>14}  {"Trunc":>5}  {"AvgT":>5}')
print('  ' + '-'*75)
rows = []
for src in ['math', 'gsm8k', 'mmlu_math (all subjects)']:
    sub = df[df['src_group']==src]
    if len(sub) == 0: continue
    n, ncorr = len(sub), sub['correct'].sum()
    acc = ncorr/n*100
    lo, hi = wilson_ci(ncorr, n)
    trunc = sub['truncated'].sum() if 'truncated' in sub.columns else 0
    avg_t = sub['time_sec'].mean()
    ci_str = f'[{lo:.1f}, {hi:.1f}]'
    rows.append({'Dataset':src,'N':n,'Correct':int(ncorr),'Accuracy%':round(acc,1),
                 '95CI_lo':lo,'95CI_hi':hi,'Truncated':int(trunc),'AvgTime_s':round(avg_t,0)})
    print(f'  {src:<38} {n:>5}  {acc:>6.1f}  {ci_str:>14}  {int(trunc):>5}  {avg_t:>5.0f}s')

# MATH by difficulty
if 'level' in df.columns:
    math_df = df[df['source']=='math']
    if len(math_df) > 0:
        print()
        print('  MATH breakdown by difficulty:')
        print(f'  {"Level":<15} {"N":>5}  {"Acc%":>6}  {"95% CI":>14}')
        print('  ' + '-'*45)
        for lvl in sorted(math_df['level'].unique()):
            sub = math_df[math_df['level']==lvl]
            n, ncorr = len(sub), sub['correct'].sum()
            acc = ncorr/n*100
            lo, hi = wilson_ci(ncorr, n)
            print(f'  {lvl:<15} {n:>5}  {acc:>6.1f}  [{lo:.1f}, {hi:.1f}]')

print('=' * 70)

# Save files
pd.DataFrame(rows).to_csv('/kaggle/working/gm_summary.csv', index=False)
df.to_csv('/kaggle/working/gm_full.csv', index=False)
with open('/kaggle/working/gm_paper_meta.json','w') as f:
    json.dump(PAPER_META, f, indent=2)

print('\n💾 Saved: gm_summary.csv, gm_full.csv, gm_paper_meta.json')
print()
print('📝 Paper Methods section (copy this):')
print(f'   Model: {PAPER_META["model_id"]} ({PAPER_META["quantization"]})')
print(f'   Hardware: {PAPER_META["hardware"]}')
print(f'   Inference: greedy decoding (T=0), context={PAPER_META["context_size"]} tokens')
print(f'   MATH: n={len(df[df["source"]=="math"])}, GSM8K: n={len(df[df["source"]=="gsm8k"])}')
print(f'   MMLU: n={len(df[df["src_group"]=="mmlu_math (all subjects)"])}')
print(f'   Random seed: {PAPER_META["rand_seed"]}. All results reproducible.')
